# Solutions — Notebook 07: Bootstrap Inference for Quantile Regression

Complete solutions to all six exercises.

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from panelbox.inference.quantile import QuantileBootstrap
from panelbox.models.quantile import PooledQuantile

warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 11,
    }
)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
PLOTS_DIR = BASE_DIR / "outputs" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


# Helper functions
def make_X(data, *cols):
    """Build design matrix with intercept."""
    arrays = [np.ones(len(data))] + [data[c].values.astype(float) for c in cols]
    return np.column_stack(arrays)


def make_fitted_model(model, result, tau_idx=0):
    """Add bootstrap-required attributes to a PooledQuantile model."""
    model.X = model.exog
    model.y = model.endog
    model.entity_ids = model.entity_id
    model.k_exog = model.exog.shape[1]
    model.nobs = len(model.endog)
    model.params = result.params[:, tau_idx] if result.params.ndim == 2 else result.params
    return model


# Load data
data = pd.read_csv(DATA_DIR / "card_education.csv")
y = data["lwage"].values
X = make_X(data, "educ", "exper")
entity_id = data["id"].values
var_names = ["const", "educ", "exper"]

# Fit baseline model
model = PooledQuantile(y, X, entity_id=entity_id, quantiles=0.5)
result = model.fit(se_type="cluster")
make_fitted_model(model, result)

print(f"Data loaded: {data.shape}")
print("Baseline median regression fitted.")

---

## Exercise 1: Naive vs Cluster Bootstrap (Easy)

Compare pairs (naive) bootstrap with cluster bootstrap.

In [ ]:
# Pairs bootstrap (ignores panel structure)
bs_pairs = QuantileBootstrap(
    model, tau=0.5, n_boot=499, method="pairs", ci_method="percentile", random_state=42
)
boot_pairs = bs_pairs.bootstrap(n_jobs=-1, verbose=False)

# Cluster bootstrap (preserves within-entity correlation)
bs_cluster = QuantileBootstrap(
    model, tau=0.5, n_boot=499, method="cluster", ci_method="percentile", random_state=42
)
boot_cluster = bs_cluster.bootstrap(n_jobs=-1, verbose=False)

# Comparison table
print("=== Exercise 1: Pairs vs Cluster Bootstrap ===")
print(f"{'Variable':<10} {'SE (pairs)':>12} {'SE (cluster)':>14} {'Ratio':>8}")
print("-" * 50)
for idx, var in enumerate(var_names):
    se_p = np.nanstd(boot_pairs.boot_params[:, idx])
    se_c = np.nanstd(boot_cluster.boot_params[:, idx])
    ratio = se_c / se_p if se_p > 0 else np.nan
    print(f"{var:<10} {se_p:12.4f} {se_c:14.4f} {ratio:8.2f}")

print()
print("Interpretation:")
print("  The cluster bootstrap SE is LARGER because it correctly accounts")
print("  for within-entity correlation. The pairs bootstrap treats each")
print("  observation as independent, which underestimates uncertainty")
print("  when the same entity contributes multiple observations.")
print("  -> Always use cluster bootstrap for panel data!")

---

## Exercise 2: CI Stability Across B (Easy)

Investigate how stable confidence intervals are as B increases.

In [ ]:
B_list = [99, 199, 499, 999, 1999]
educ_idx = 1  # education coefficient

ci_lower_list = []
ci_upper_list = []
se_list = []

print("=== Exercise 2: CI Stability ===")
print(f"{'B':>6} {'SE(educ)':>10} {'CI_lower':>10} {'CI_upper':>10} {'CI_width':>10}")
print("-" * 52)

for B in B_list:
    bs_temp = QuantileBootstrap(
        model, tau=0.5, n_boot=B, method="cluster", ci_method="percentile", random_state=42
    )
    boot_temp = bs_temp.bootstrap(n_jobs=-1, verbose=False)

    se = np.nanstd(boot_temp.boot_params[:, educ_idx])
    ci_lo = boot_temp.ci_lower[educ_idx]
    ci_hi = boot_temp.ci_upper[educ_idx]

    se_list.append(se)
    ci_lower_list.append(ci_lo)
    ci_upper_list.append(ci_hi)

    print(f"{B:6d} {se:10.4f} {ci_lo:10.4f} {ci_hi:10.4f} {ci_hi - ci_lo:10.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(
    B_list, ci_lower_list, ci_upper_list, alpha=0.3, color="steelblue", label="95% CI bounds"
)
ax.plot(B_list, ci_lower_list, "v-", color="steelblue", markersize=8, label="CI lower")
ax.plot(B_list, ci_upper_list, "^-", color="steelblue", markersize=8, label="CI upper")
ax.axhline(
    result.params.ravel()[educ_idx],
    color="red",
    linestyle="--",
    linewidth=2,
    label="Point estimate",
)

ax.set_xlabel("Number of Bootstrap Replications (B)", fontsize=12)
ax.set_ylabel("Education Coefficient", fontsize=12)
ax.set_title("Confidence Interval Stability as B Increases", fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print()
print("Conclusion:")
print("  CIs typically stabilize around B=499-999.")
print("  B=99 may produce unstable CI bounds.")
print("  B=999 is a good default for standard analysis.")

---

## Exercise 3: Compare CI Methods (Medium)

Compare percentile, normal, and BCa confidence intervals.

In [ ]:
ci_methods = ["percentile", "normal", "bca"]
results_ci = {}

for method in ci_methods:
    bs_temp = QuantileBootstrap(
        model, tau=0.5, n_boot=999, method="cluster", ci_method=method, random_state=42
    )
    boot_temp = bs_temp.bootstrap(n_jobs=-1, verbose=False)
    results_ci[method] = boot_temp

# Comparison table
print("=== Exercise 3: CI Method Comparison (tau=0.5, B=999) ===")
print()

for var_idx, var in enumerate(var_names):
    coef = result.params.ravel()[var_idx]
    print(f"{var} (estimate = {coef:.4f}):")
    print(f"  {'Method':<12} {'CI Lower':>10} {'CI Upper':>10} {'Width':>10}")
    print(f"  {'-' * 45}")
    for method in ci_methods:
        ci_lo = results_ci[method].ci_lower[var_idx]
        ci_hi = results_ci[method].ci_upper[var_idx]
        width = ci_hi - ci_lo
        print(f"  {method:<12} {ci_lo:10.4f} {ci_hi:10.4f} {width:10.4f}")
    print()

print("Interpretation:")
print("  - Normal CIs are symmetric around the estimate.")
print("  - Percentile CIs use bootstrap distribution quantiles (may be asymmetric).")
print("  - BCa CIs adjust for bias and skewness (most accurate in theory).")
print("  - Note: Current PanelBox BCa implementation falls back to percentile.")
print("  - For most applications, percentile CIs are sufficient.")

---

## Exercise 4: Multi-Quantile Bootstrap Process (Medium)

Run bootstrap at multiple quantiles and test for non-flat coefficient path.

In [ ]:
taus = [0.1, 0.25, 0.5, 0.75, 0.9]
educ_idx = 1

coefs_educ = []
ci_lo_educ = []
ci_hi_educ = []
boot_arrays = {}

print("=== Exercise 4: Multi-Quantile Bootstrap ===")
print(f"Estimating at tau = {taus} with B=499 each...\n")

for tau in taus:
    m = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
    r = m.fit(se_type="cluster")
    make_fitted_model(m, r)

    bs_temp = QuantileBootstrap(
        m, tau=tau, n_boot=499, method="cluster", ci_method="percentile", random_state=42
    )
    boot_temp = bs_temp.bootstrap(n_jobs=-1, verbose=False)

    coef = r.params.ravel()[educ_idx]
    coefs_educ.append(coef)
    ci_lo_educ.append(boot_temp.ci_lower[educ_idx])
    ci_hi_educ.append(boot_temp.ci_upper[educ_idx])
    boot_arrays[tau] = boot_temp.boot_params

    valid = (~np.any(np.isnan(boot_temp.boot_params), axis=1)).sum()
    print(
        f"  tau={tau:.2f}: beta_educ={coef:.4f}  "
        f"CI=[{boot_temp.ci_lower[educ_idx]:.4f}, {boot_temp.ci_upper[educ_idx]:.4f}]  "
        f"({valid}/499 valid)"
    )

# Plot quantile process with bootstrap CIs
fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(
    taus, ci_lo_educ, ci_hi_educ, alpha=0.25, color="steelblue", label="95% CI (bootstrap)"
)
ax.plot(taus, coefs_educ, "o-", color="steelblue", linewidth=2.5, markersize=9, label="QR estimate")

# OLS baseline
from numpy.linalg import lstsq

beta_ols = lstsq(X, y, rcond=None)[0]
ax.axhline(
    beta_ols[educ_idx],
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"OLS = {beta_ols[educ_idx]:.4f}",
)

ax.set_xlabel("Quantile (tau)", fontsize=12)
ax.set_ylabel("Education Coefficient", fontsize=12)
ax.set_title("Quantile Process with Bootstrap Confidence Bands", fontsize=13)
ax.set_xticks(taus)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Test for non-flat path: beta(0.1) vs beta(0.9)
valid_10 = ~np.any(np.isnan(boot_arrays[0.1]), axis=1)
valid_90 = ~np.any(np.isnan(boot_arrays[0.9]), axis=1)
valid_both = valid_10 & valid_90

diff_boot = boot_arrays[0.9][valid_both, educ_idx] - boot_arrays[0.1][valid_both, educ_idx]
obs_diff = coefs_educ[-1] - coefs_educ[0]  # tau=0.9 - tau=0.1

print("\n=== Test: Is the coefficient path non-flat? ===")
print(f"beta_educ(0.9) - beta_educ(0.1) = {obs_diff:.4f}")

if len(diff_boot) == 0 or diff_boot.shape[0] == 0:
    print("Bootstrap failed - no valid paired replications for tau=0.1 and tau=0.9.")
    print("Falling back to nanpercentile on all bootstrap draws...")
    diff_boot_all = boot_arrays[0.9][:, educ_idx] - boot_arrays[0.1][:, educ_idx]
    ci_diff_lo = np.nanpercentile(diff_boot_all, 2.5)
    ci_diff_hi = np.nanpercentile(diff_boot_all, 97.5)
    print(f"Bootstrap CI for difference (nanpercentile): [{ci_diff_lo:.4f}, {ci_diff_hi:.4f}]")
    if np.isnan(ci_diff_lo) or np.isnan(ci_diff_hi):
        print("-> All bootstrap replications failed. Cannot test.")
    elif ci_diff_lo > 0 or ci_diff_hi < 0:
        print("-> Significantly non-flat: education effects differ across quantiles.")
    else:
        print("-> Not significantly non-flat at 5% level.")
else:
    ci_diff_lo = np.percentile(diff_boot, 2.5)
    ci_diff_hi = np.percentile(diff_boot, 97.5)
    print(f"Bootstrap CI for difference: [{ci_diff_lo:.4f}, {ci_diff_hi:.4f}]")
    print(f"({len(diff_boot)} valid paired replications)")

    if ci_diff_lo > 0 or ci_diff_hi < 0:
        print("-> Significantly non-flat: education effects differ across quantiles.")
    else:
        print("-> Not significantly non-flat at 5% level.")

---

## Exercise 5: Coverage Probability Simulation (Hard)

Verify that bootstrap CIs achieve ~95% nominal coverage.

In [ ]:
def generate_panel_data(N=50, T=5, beta=None, seed=None):
    """Generate panel data with known coefficients."""
    if seed is not None:
        np.random.seed(seed)
    if beta is None:
        beta = np.array([1.0, 0.5, -0.2])

    rows = []
    for i in range(N):
        x1_i = np.random.normal(0, 1, T)
        x2_i = np.random.normal(0, 1, T)
        alpha_i = np.random.normal(0, 0.5)  # entity effect
        eps = np.random.normal(0, 0.5, T)
        y_i = beta[0] + beta[1] * x1_i + beta[2] * x2_i + alpha_i + eps
        for t in range(T):
            rows.append({"entity": i, "time": t, "y": y_i[t], "x1": x1_i[t], "x2": x2_i[t]})
    return pd.DataFrame(rows)


# Run coverage simulation
beta_true = np.array([1.0, 0.5, -0.2])
N_sim = 200
B_boot = 199
coverage = np.zeros((N_sim, 3))

print("=== Exercise 5: Coverage Probability Simulation ===")
print(f"N_sim={N_sim}, B={B_boot}, N=50, T=5")
print(f"True beta = {beta_true}")
print("Running simulations...")

t0 = time.time()
for sim in range(N_sim):
    df_sim = generate_panel_data(N=50, T=5, beta=beta_true, seed=sim)
    X_sim = make_X(df_sim, "x1", "x2")
    y_sim = df_sim["y"].values
    eid_sim = df_sim["entity"].values

    m_sim = PooledQuantile(y_sim, X_sim, entity_id=eid_sim, quantiles=0.5)
    r_sim = m_sim.fit(se_type="cluster")
    make_fitted_model(m_sim, r_sim)

    bs_sim = QuantileBootstrap(
        m_sim, tau=0.5, n_boot=B_boot, method="cluster", ci_method="percentile", random_state=sim
    )
    boot_sim = bs_sim.bootstrap(n_jobs=-1, verbose=False)

    for j in range(3):
        coverage[sim, j] = boot_sim.ci_lower[j] <= beta_true[j] <= boot_sim.ci_upper[j]

    if (sim + 1) % 50 == 0:
        print(f"  Completed {sim + 1}/{N_sim}...")

elapsed = time.time() - t0
print(f"\nCompleted in {elapsed:.1f}s")

# Report coverage
print(f"\n{'Variable':<10} {'True Beta':>10} {'Coverage':>10} {'Nominal':>10}")
print("-" * 45)
var_names_sim = ["const", "x1", "x2"]
for j, var in enumerate(var_names_sim):
    cov_rate = coverage[:, j].mean()
    print(f"{var:<10} {beta_true[j]:10.2f} {cov_rate:10.1%} {'95%':>10}")

print()
print("Interpretation:")
print("  Coverage should be close to 95%.")
print("  Some under-coverage is expected with B=199 (increase B for better accuracy).")
print("  Over-coverage may indicate conservative CIs.")

---

## Exercise 6: Glass Ceiling Hypothesis (Hard)

Test whether returns to education are significantly higher at the top of the wage distribution.

In [ ]:
taus_gc = [0.1, 0.25, 0.5, 0.75, 0.9]
educ_idx = 1

results_gc = {}
boots_gc = {}

print("=== Exercise 6: Glass Ceiling Hypothesis ===")
print("Estimating quantile coefficients with bootstrap...\n")

for tau in taus_gc:
    m = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
    r = m.fit(se_type="cluster")
    make_fitted_model(m, r)
    results_gc[tau] = r

    bs_temp = QuantileBootstrap(
        m, tau=tau, n_boot=999, method="cluster", ci_method="percentile", random_state=42
    )
    boots_gc[tau] = bs_temp.bootstrap(n_jobs=-1, verbose=False)

    coef = r.params.ravel()[educ_idx]
    se = np.nanstd(boots_gc[tau].boot_params[:, educ_idx])
    print(
        f"  tau={tau:.2f}: beta_educ={coef:.4f}  SE={se:.4f}  "
        f"CI=[{boots_gc[tau].ci_lower[educ_idx]:.4f}, "
        f"{boots_gc[tau].ci_upper[educ_idx]:.4f}]"
    )

# One-sided test: H0: beta(0.9) <= beta(0.1)
valid_10 = ~np.any(np.isnan(boots_gc[0.1].boot_params), axis=1)
valid_90 = ~np.any(np.isnan(boots_gc[0.9].boot_params), axis=1)
valid_both = valid_10 & valid_90

diff_boot = (
    boots_gc[0.9].boot_params[valid_both, educ_idx]
    - boots_gc[0.1].boot_params[valid_both, educ_idx]
)

obs_diff = results_gc[0.9].params.ravel()[educ_idx] - results_gc[0.1].params.ravel()[educ_idx]

print("\n=== One-Sided Test: H0: beta_educ(0.9) <= beta_educ(0.1) ===")
print(f"beta_educ(0.1) = {results_gc[0.1].params.ravel()[educ_idx]:.4f}")
print(f"beta_educ(0.9) = {results_gc[0.9].params.ravel()[educ_idx]:.4f}")
print(f"Observed difference: {obs_diff:.4f}")

if len(diff_boot) == 0 or diff_boot.shape[0] == 0:
    print("Bootstrap failed - no valid paired replications for tau=0.1 and tau=0.9.")
    print("Falling back to nanpercentile on all bootstrap draws...")
    diff_boot_all = boots_gc[0.9].boot_params[:, educ_idx] - boots_gc[0.1].boot_params[:, educ_idx]
    n_valid_fallback = np.sum(~np.isnan(diff_boot_all))
    if n_valid_fallback == 0:
        print("-> All bootstrap replications failed. Cannot compute p-value.")
    else:
        p_value_one_sided = np.nanmean(diff_boot_all <= 0)
        print(f"One-sided p-value (nanmean, {n_valid_fallback} valid): {p_value_one_sided:.4f}")
        if p_value_one_sided < 0.05:
            print("-> REJECT H0: Returns to education are significantly higher at the top.")
            print('   This supports the "glass ceiling" hypothesis.')
        else:
            print("-> FAIL TO REJECT H0: No significant evidence of higher returns at the top.")
else:
    # One-sided p-value: proportion of bootstrap diffs <= 0
    p_value_one_sided = np.mean(diff_boot <= 0)
    print(f"One-sided p-value: {p_value_one_sided:.4f}")
    print(f"({len(diff_boot)} valid paired replications)")

    if p_value_one_sided < 0.05:
        print("-> REJECT H0: Returns to education are significantly higher at the top.")
        print('   This supports the "glass ceiling" hypothesis.')
    else:
        print("-> FAIL TO REJECT H0: No significant evidence of higher returns at the top.")

In [ ]:
# Forest plot
fig, ax = plt.subplots(figsize=(8, 6))

coefs_gc = [results_gc[t].params.ravel()[educ_idx] for t in taus_gc]
ci_lo_gc = [boots_gc[t].ci_lower[educ_idx] for t in taus_gc]
ci_hi_gc = [boots_gc[t].ci_upper[educ_idx] for t in taus_gc]

y_pos = np.arange(len(taus_gc))
errors = [[c - l for c, l in zip(coefs_gc, ci_lo_gc)], [h - c for c, h in zip(coefs_gc, ci_hi_gc)]]

ax.errorbar(
    coefs_gc,
    y_pos,
    xerr=errors,
    fmt="o",
    markersize=10,
    capsize=8,
    linewidth=2.5,
    color="steelblue",
    ecolor="steelblue",
)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"tau = {t}" for t in taus_gc], fontsize=11)
ax.axvline(0, color="red", linestyle="--", alpha=0.6, linewidth=1.5)

# OLS baseline
beta_ols_val = lstsq(X, y, rcond=None)[0][educ_idx]
ax.axvline(
    beta_ols_val, color="green", linestyle=":", linewidth=2, label=f"OLS = {beta_ols_val:.4f}"
)

ax.set_xlabel("Education Coefficient", fontsize=12)
ax.set_title(
    "Glass Ceiling Test: Returns to Education by Quantile\n"
    "(Points = estimates, bars = 95% bootstrap CI)",
    fontsize=12,
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print()
print("INTERPRETATION:")
print("  If education coefficients increase with tau, this suggests")
print("  education is more rewarding for those already at the top of")
print("  the wage distribution (consistent with a glass ceiling for")
print("  workers without education). If they decrease, the opposite")
print("  pattern holds — education lifts the bottom more than the top.")